In [3]:
import torch
import torchaudio
import torchaudio.transforms as T
import matplotlib.pyplot as plt

## Conversión de Audio a Mel-Spectrogramas en GPU

Este bloque procesa todos los archivos `.wav` de la carpeta `archive` y sus subdirectorios (clases), convirtiéndolos a imágenes mel-espectrogram que se guardan en `archive_img` con la misma estructura de carpetas pero con sufijo `_img`.

In [ ]:
from pathlib import Path
from scipy.io import wavfile
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Encontrar la carpeta raíz del proyecto
def find_project_root(start: Path) -> Path:
    """Busca hacia arriba la carpeta que contiene `archive` y `archive_img`."""
    for p in [start, *start.parents]:
        if (p / "archive").exists() and (p / "archive_img").exists():
            return p
    return start

base_dir = find_project_root(Path.cwd())
source_root = base_dir / "archive"
target_root = base_dir / "archive_img"

print(f"Origen: {source_root}")
print(f"Destino: {target_root}")

# Validar carpetas
if not source_root.exists() or not target_root.exists():
    raise FileNotFoundError(f"No se encontraron las carpetas esperadas")

# GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}\n")

# Mel-Spectrogram en GPU
mel_transform = T.MelSpectrogram(
    sample_rate=16000, n_fft=1024, hop_length=256, n_mels=128
).to(device)

def load_wav_as_tensor(audio_path: Path):
    """Carga WAV sin dependencias problemáticas."""
    sr, data = wavfile.read(str(audio_path))
    if data.dtype.kind in {"i", "u"}:
        max_val = float(max(abs(data.min()), abs(data.max()), 1))
        data = data.astype("float32") / max_val
    else:
        data = data.astype("float32")
    if data.ndim == 1:
        data = data[None, :]
    else:
        data = data.T
    return torch.from_numpy(data), sr

def norm_to_uint8(arr):
    """Normaliza array a uint8 para guardar como imagen."""
    arr = np.clip(arr, 0, None)
    arr = (arr / (arr.max() + 1e-8) * 255).astype("uint8")
    return arr

saved_count = 0
skipped_count = 0

for class_dir in sorted(source_root.iterdir()):
    if not class_dir.is_dir() or class_dir.name.startswith("."):
        continue
    
    target_class_dir = target_root / f"{class_dir.name}_img"
    target_class_dir.mkdir(parents=True, exist_ok=True)
    
    wav_files = list(class_dir.glob("*.wav"))
    if len(wav_files) > 0:
        print(f"{class_dir.name}: {len(wav_files)} archivos")
    
    for audio_path in wav_files:
        try:
            waveform, sr = load_wav_as_tensor(audio_path)
            if waveform.size(0) > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
            
            waveform = waveform.to(device)
            if sr != 16000:
                resampler = T.Resample(orig_freq=sr, new_freq=16000).to(device)
                waveform = resampler(waveform)
            
            mel = mel_transform(waveform)
            mel_db = torch.log10(mel + 1e-9).squeeze(0).detach().cpu().numpy()
            mel_img = norm_to_uint8(mel_db)
            
            out_path = target_class_dir / (audio_path.stem + ".png")
            plt.imsave(str(out_path), mel_img, cmap="magma")
            
            saved_count += 1
        except Exception as e:
            skipped_count += 1

print(f"\n{'='*50}")
print(f"Imágenes guardadas: {saved_count}")
print(f"Omitidos: {skipped_count}")
print(f"{'='*50}")

Origen: c:\Users\boyfa\OneDrive\Documentos\AI\Proyecto1\repo\AI_Proyecto1_2026\archive
Destino: c:\Users\boyfa\OneDrive\Documentos\AI\Proyecto1\repo\AI_Proyecto1_2026\archive_img
Dispositivo: cuda

_background_noise_: 6 archivos
backward: 1664 archivos
bed: 2014 archivos
bird: 2064 archivos
cat: 2031 archivos
dog: 2128 archivos
down: 3917 archivos
eight: 3787 archivos
five: 4052 archivos
follow: 1579 archivos
forward: 1557 archivos
four: 3728 archivos
go: 3880 archivos
happy: 2054 archivos
house: 2113 archivos
learn: 1575 archivos
left: 3801 archivos
marvin: 2100 archivos
nine: 3934 archivos
no: 3941 archivos
off: 3745 archivos
on: 3845 archivos
one: 3890 archivos
right: 3778 archivos
seven: 3998 archivos
sheila: 2022 archivos
six: 3860 archivos
stop: 3872 archivos
three: 3727 archivos
tree: 1759 archivos
two: 3880 archivos
up: 3723 archivos
visual: 1592 archivos
wow: 2123 archivos
yes: 4044 archivos
zero: 4052 archivos

✓ Imágenes guardadas: 105835
✗ Omitidos: 0
